In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import randint
from sklearn.metrics import accuracy_score
import joblib

In [2]:
#Import and Split Data

# Adjust pandas display options to show all columns
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)

# Load the original data
data_original = pd.read_csv('Kaggle_Sirio_Libanes_ICU_Prediction_Modded_Final.csv')

data_train = data_original.iloc[:1295].reset_index(drop=True)
data_test = data_original.iloc[1295:].reset_index(drop=True)

print(data_test.dtypes)
print(data_train.shape, data_test.shape)
display(data_test.head(6))

AGE_ABOVE65                           int64
GENDER                                int64
DISEASE GROUPING 1                  float64
DISEASE GROUPING 2                  float64
DISEASE GROUPING 3                  float64
DISEASE GROUPING 4                  float64
DISEASE GROUPING 5                  float64
DISEASE GROUPING 6                  float64
HTN                                 float64
IMMUNOCOMPROMISED                   float64
OTHER                               float64
AGE_PERCENTILE_NORM                 float64
WINDOW_NORM                         float64
BLOODPRESSURE_DIASTOLIC_MEAN        float64
BLOODPRESSURE_SISTOLIC_MEAN         float64
HEART_RATE_MEAN                     float64
RESPIRATORY_RATE_MEAN               float64
TEMPERATURE_MEAN                    float64
OXYGEN_SATURATION_MEAN              float64
BLOODPRESSURE_DIASTOLIC_MEDIAN      float64
BLOODPRESSURE_SISTOLIC_MEDIAN       float64
HEART_RATE_MEDIAN                   float64
RESPIRATORY_RATE_MEDIAN         

,AGE_ABOVE65,GENDER,DISEASE GROUPING 1,DISEASE GROUPING 2,DISEASE GROUPING 3,DISEASE GROUPING 4,DISEASE GROUPING 5,DISEASE GROUPING 6,HTN,IMMUNOCOMPROMISED,OTHER,AGE_PERCENTILE_NORM,WINDOW_NORM,BLOODPRESSURE_DIASTOLIC_MEAN,BLOODPRESSURE_SISTOLIC_MEAN,HEART_RATE_MEAN,RESPIRATORY_RATE_MEAN,TEMPERATURE_MEAN,OXYGEN_SATURATION_MEAN,BLOODPRESSURE_DIASTOLIC_MEDIAN,BLOODPRESSURE_SISTOLIC_MEDIAN,HEART_RATE_MEDIAN,RESPIRATORY_RATE_MEDIAN,TEMPERATURE_MEDIAN,OXYGEN_SATURATION_MEDIAN,BLOODPRESSURE_DIASTOLIC_MIN,BLOODPRESSURE_SISTOLIC_MIN,HEART_RATE_MIN,RESPIRATORY_RATE_MIN,TEMPERATURE_MIN,OXYGEN_SATURATION_MIN,BLOODPRESSURE_DIASTOLIC_MAX,BLOODPRESSURE_SISTOLIC_MAX,HEART_RATE_MAX,RESPIRATORY_RATE_MAX,TEMPERATURE_MAX,OXYGEN_SATURATION_MAX,BLOODPRESSURE_DIASTOLIC_DIFF,BLOODPRESSURE_SISTOLIC_DIFF,HEART_RATE_DIFF,RESPIRATORY_RATE_DIFF,TEMPERATURE_DIFF,OXYGEN_SATURATION_DIFF,BLOODPRESSURE_DIASTOLIC_DIFF_REL,BLOODPRESSURE_SISTOLIC_DIFF_REL,HEART_RATE_DIFF_REL,RESPIRATORY_RATE_DIFF_REL,TEMPERATURE_DIFF_REL,OXYGEN_SATURATION_DIFF_REL,ICU
0,0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.2,0.066667,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0
1,0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.2,0.200000,0.086420,-0.384615,-0.075472,-0.457627,0.380952,0.789474,0.086420,-0.384615,-0.075472,-0.448276,0.357143,0.789474,0.237113,-0.125,0.025641,-0.357143,0.604396,0.919192,-0.247863,-0.567568,-0.268657,-0.515152,0.159420,0.789474,-1.000000,-1.000000,-1.000000,-1.000000,-0.952381,-1.000000,-1.000000,-1.000000,-1.000000,-1.000000,-0.953785,-1.000000,0
2,0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.2,0.333333,NaN,NaN,NaN,NaN,0.142857,0.789474,NaN,NaN,NaN,NaN,0.142857,0.789474,NaN,NaN,NaN,NaN,0.406593,0.919192,NaN,NaN,NaN,NaN,0.014493,0.789474,NaN,NaN,NaN,NaN,-0.857143,-1.000000,NaN,NaN,NaN,NaN,-0.859089,-1.000000,0
3,0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.2,0.600000,0.333333,-0.384615,-0.207547,-0.728814,0.035714,0.736842,0.333333,-0.384615,-0.207547,-0.724138,0.035714,0.736842,0.443299,-0.125,-0.094017,-0.642857,0.406593,0.898990,-0.076923,-0.567568,-0.373134,-0.757576,-0.159420,0.736842,-1.000000,-1.000000,-1.000000,-1.000000,-1.000000,-1.000000,-1.000000,-1.000000,-1.000000,-1.000000,-1.000000,-1.000000,0
4,0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.2,1.000000,0.060442,-0.304006,-0.345519,-0.302966,0.167039,0.684759,0.086420,-0.261538,-0.358491,-0.448276,0.178571,0.736842,-0.154639,-0.375,-0.658120,-0.642857,0.274725,0.676768,0.094017,-0.156757,-0.119403,0.696970,0.478261,0.894737,-0.321739,-0.288344,-0.236641,0.411765,-0.333333,-0.717172,-0.491304,-0.288381,-0.409669,0.385399,-0.344203,-0.717417,1
5,0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.4,0.066667,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0


In [3]:
# Correct column dtypes

# Define a function to handle the conversion
def convert_columns_to_int64(df, cols):
    for col in cols:
        df[col] = df[col].fillna(999).astype('Int64').replace(999, pd.NA)
    return df

# Specify the columns to convert
columns_to_convert = data_test.columns[2:11]

# Convert the column dtypes to Int64 for both dataframes
data_test = convert_columns_to_int64(data_test, columns_to_convert)
data_train = convert_columns_to_int64(data_train, columns_to_convert)

# Verify the changes
print(data_test.dtypes)
print(data_train.dtypes)

AGE_ABOVE65                           int64
GENDER                                int64
DISEASE GROUPING 1                    Int64
DISEASE GROUPING 2                    Int64
DISEASE GROUPING 3                    Int64
DISEASE GROUPING 4                    Int64
DISEASE GROUPING 5                    Int64
DISEASE GROUPING 6                    Int64
HTN                                   Int64
IMMUNOCOMPROMISED                     Int64
OTHER                                 Int64
AGE_PERCENTILE_NORM                 float64
WINDOW_NORM                         float64
BLOODPRESSURE_DIASTOLIC_MEAN        float64
BLOODPRESSURE_SISTOLIC_MEAN         float64
HEART_RATE_MEAN                     float64
RESPIRATORY_RATE_MEAN               float64
TEMPERATURE_MEAN                    float64
OXYGEN_SATURATION_MEAN              float64
BLOODPRESSURE_DIASTOLIC_MEDIAN      float64
BLOODPRESSURE_SISTOLIC_MEDIAN       float64
HEART_RATE_MEDIAN                   float64
RESPIRATORY_RATE_MEDIAN         

In [4]:
# Separate features and target
X = data_train.drop(columns=['ICU'])
y = data_train['ICU']

# Split into training/validation
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42)

# Display the shapes of the resulting datasets to ensure correct splitting
X_train.shape, X_test.shape, y_train.shape, y_test.shape

((1036, 49), (259, 49), (1036,), (259,))

In [5]:
#Idenitfy column types
float64_columns = X_train.select_dtypes(include=['float64']).columns.tolist()
binary_columns = X_train.select_dtypes(include=['int64']).columns.tolist()

print("Float64 Columns:", float64_columns)
print("Binary Columns:", binary_columns)

Float64 Columns: ['AGE_PERCENTILE_NORM', 'WINDOW_NORM', 'BLOODPRESSURE_DIASTOLIC_MEAN', 'BLOODPRESSURE_SISTOLIC_MEAN', 'HEART_RATE_MEAN', 'RESPIRATORY_RATE_MEAN', 'TEMPERATURE_MEAN', 'OXYGEN_SATURATION_MEAN', 'BLOODPRESSURE_DIASTOLIC_MEDIAN', 'BLOODPRESSURE_SISTOLIC_MEDIAN', 'HEART_RATE_MEDIAN', 'RESPIRATORY_RATE_MEDIAN', 'TEMPERATURE_MEDIAN', 'OXYGEN_SATURATION_MEDIAN', 'BLOODPRESSURE_DIASTOLIC_MIN', 'BLOODPRESSURE_SISTOLIC_MIN', 'HEART_RATE_MIN', 'RESPIRATORY_RATE_MIN', 'TEMPERATURE_MIN', 'OXYGEN_SATURATION_MIN', 'BLOODPRESSURE_DIASTOLIC_MAX', 'BLOODPRESSURE_SISTOLIC_MAX', 'HEART_RATE_MAX', 'RESPIRATORY_RATE_MAX', 'TEMPERATURE_MAX', 'OXYGEN_SATURATION_MAX', 'BLOODPRESSURE_DIASTOLIC_DIFF', 'BLOODPRESSURE_SISTOLIC_DIFF', 'HEART_RATE_DIFF', 'RESPIRATORY_RATE_DIFF', 'TEMPERATURE_DIFF', 'OXYGEN_SATURATION_DIFF', 'BLOODPRESSURE_DIASTOLIC_DIFF_REL', 'BLOODPRESSURE_SISTOLIC_DIFF_REL', 'HEART_RATE_DIFF_REL', 'RESPIRATORY_RATE_DIFF_REL', 'TEMPERATURE_DIFF_REL', 'OXYGEN_SATURATION_DIFF_REL']
Bi

In [6]:
# Define the preprocessing and modeling pipeline with correct imputation strategies
preprocessor = ColumnTransformer(
    transformers=[
        ('num', Pipeline(steps=[('imputer', SimpleImputer(strategy='mean'))]), float64_columns),
        ('bin', Pipeline(steps=[('imputer', SimpleImputer(strategy='most_frequent'))]), binary_columns),
    ])

In [7]:
#Test imputed (filled-in) data with Random forest
rf_pipeline_base = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', RandomForestClassifier(random_state=42))
])

# Train the model
rf_pipeline_base.fit(X_train, y_train)

# Evaluate model
y_pred = rf_pipeline_base.predict(X_test)

rf_pipeline_accuracy = accuracy_score(y_test, y_pred)

In [8]:
#Test Grid Search Tuned RF

# Define the parameter grid
param_grid = {
    'classifier__n_estimators': [100, 200, 300],
    'classifier__max_depth': [None, 10, 20, 30],
    'classifier__min_samples_split': [2, 5, 10],
    'classifier__min_samples_leaf': [1, 2, 4],
    'classifier__max_features': ['auto', 'sqrt', 'log2']
}

rf_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', RandomForestClassifier(random_state=42))
])

# Initialize Grid Search
rf_gs = GridSearchCV(estimator=rf_pipeline, param_grid=param_grid, cv=5, n_jobs=-1, verbose=2)

# Train the model
rf_gs.fit(X_train, y_train)

# Evaluate model
y_pred = rf_gs.predict(X_test)

rf_gs_accuracy = accuracy_score(y_test, y_pred)

Fitting 5 folds for each of 324 candidates, totalling 1620 fits


c:\Users\User\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\model_selection\_validation.py:540: FitFailedWarning: 
540 fits failed out of a total of 1620.
The score on these train-test partitions for these parameters will be set to nan.
If these failures are not expected, you can try to debug them by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
291 fits failed with the following error:
Traceback (most recent call last):
  File "c:\Users\User\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\model_selection\_validation.py", line 888, in _fit_and_score
    estimator.fit(X_train, y_train, **fit_params)
  File "c:\Users\User\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\base.py", line 1473, in wrapper
    return fit_method(estimator, *args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\User\AppDat

In [9]:
#Test Random Search Tuned RF

# Define the parameter distribution
param_dist = {
    'classifier__n_estimators': randint(100, 500),
    'classifier__max_depth': [None, 10, 20, 30, 40, 50],
    'classifier__min_samples_split': randint(2, 11),
    'classifier__min_samples_leaf': randint(1, 11),
    'classifier__max_features': ['auto', 'sqrt', 'log2']
}

rf_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', RandomForestClassifier(random_state=42))
])

# Initialize Random Search
rf_rs = RandomizedSearchCV(estimator=rf_pipeline, param_distributions=param_dist, n_iter=100, cv=5, random_state=42, n_jobs=-1, verbose=2)

# Train the model
rf_rs.fit(X_train, y_train)

# Evaluate model
y_pred = rf_rs.predict(X_test)

rf_rs_accuracy = accuracy_score(y_test, y_pred)

Fitting 5 folds for each of 100 candidates, totalling 500 fits


c:\Users\User\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\model_selection\_validation.py:540: FitFailedWarning: 
160 fits failed out of a total of 500.
The score on these train-test partitions for these parameters will be set to nan.
If these failures are not expected, you can try to debug them by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
102 fits failed with the following error:
Traceback (most recent call last):
  File "c:\Users\User\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\model_selection\_validation.py", line 888, in _fit_and_score
    estimator.fit(X_train, y_train, **fit_params)
  File "c:\Users\User\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\base.py", line 1473, in wrapper
    return fit_method(estimator, *args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\User\AppData

In [10]:
#Test Logistic Regression

# Define the preprocessing and modeling pipeline for Logistic Regression
logistic_pipeline_base = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression(random_state=42, max_iter=1000))
])

# Train the Logistic Regression model
logistic_pipeline_base.fit(X_train, y_train)

# Evaluate on final test set
y_pred = logistic_pipeline_base.predict(X_test)

logistic_pipeline_accuracy = accuracy_score(y_test, y_pred)

In [11]:
#Test a Tuned Logistic Regression model

# Define the parameter grid
param_grid = {
    'classifier__C': [0.01, 0.1, 1, 10, 100],
    'classifier__penalty': ['l1', 'l2', 'elasticnet'],
    'classifier__solver': ['liblinear', 'saga', 'lbfgs']
}

# Define the preprocessing and modeling pipeline for Logistic Regression
logistic_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression(random_state=42, max_iter=10000))
])

# Initialize Grid Search
lr_gs = GridSearchCV(estimator=logistic_pipeline, param_grid=param_grid, cv=5, n_jobs=-1, verbose=2)

# Fit Grid Search
lr_gs.fit(X_train, y_train)

# Evaluate on final test set
y_pred = lr_gs.predict(X_test)

lr_gs_accuracy = accuracy_score(y_test, y_pred)

Fitting 5 folds for each of 45 candidates, totalling 225 fits


c:\Users\User\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\model_selection\_validation.py:540: FitFailedWarning: 
100 fits failed out of a total of 225.
The score on these train-test partitions for these parameters will be set to nan.
If these failures are not expected, you can try to debug them by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
25 fits failed with the following error:
Traceback (most recent call last):
  File "c:\Users\User\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\model_selection\_validation.py", line 888, in _fit_and_score
    estimator.fit(X_train, y_train, **fit_params)
  File "c:\Users\User\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\base.py", line 1473, in wrapper
    return fit_method(estimator, *args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\User\AppData\

In [12]:
#Test Decision Tree

dt_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', DecisionTreeClassifier(random_state=42))
])

# Train the model
dt_pipeline.fit(X_train, y_train)

# Evaluate model
y_pred = dt_pipeline.predict(X_test)

dt_pipeline_accuracy = accuracy_score(y_test, y_pred)

In [13]:
#Compare all models
print('random forest accuracy: ',rf_pipeline_accuracy,'\n',
      'random forest grid search accuracy: ',rf_gs_accuracy,'\n',
      'random forest random search accuracy: ',rf_rs_accuracy,'\n',
      'logistic regression accuracy: ',logistic_pipeline_accuracy,'\n',
      'logistic regression grid search accuracy: ',lr_gs_accuracy,'\n',
      'decision tree accuracy: ',dt_pipeline_accuracy)

random forest accuracy:  0.8648648648648649 
 random forest grid search accuracy:  0.8803088803088803 
 random forest random search accuracy:  0.888030888030888 
 logistic regression accuracy:  0.861003861003861 
 logistic regression grid search accuracy:  0.8687258687258688 
 decision tree accuracy:  0.7992277992277992


In [14]:
#Test all models on data_test:

X_new = data_test.drop(columns=['ICU'])
y_new = data_test['ICU']

y_new_rf = rf_pipeline_base.predict(X_new)
y_new_rfgs = rf_gs.predict(X_new)
y_new_rfrs = rf_rs.predict(X_new)
y_new_lr = logistic_pipeline_base.predict(X_new)
y_new_lrgs = lr_gs.predict(X_new)
y_new_dt = dt_pipeline.predict(X_new)

accuracy_new_rf = accuracy_score(y_new, y_new_rf)
accuracy_new_rfgs = accuracy_score(y_new, y_new_rfgs)
accuracy_new_rfrs = accuracy_score(y_new, y_new_rfrs)
accuracy_new_lr = accuracy_score(y_new, y_new_lr)
accuracy_new_lrgs = accuracy_score(y_new, y_new_lrgs)
accuracy_new_dt = accuracy_score(y_new, y_new_dt)

print('data_test random forest accuracy: ',accuracy_new_rf,'\n',
      'data_test random forest grid search accuracy: ',accuracy_new_rfgs,'\n',
      'data_test random forest random search accuracy: ',accuracy_new_rfrs,'\n',
      'data_test logistic regression accuracy: ',accuracy_new_lr,'\n',
      'data_test logistic regression grid search accuracy: ',accuracy_new_lrgs,'\n',
      'data_test decision tree accuracy: ',accuracy_new_dt)

data_test random forest accuracy:  0.8793650793650793 
 data_test random forest grid search accuracy:  0.8857142857142857 
 data_test random forest random search accuracy:  0.8825396825396825 
 data_test logistic regression accuracy:  0.8698412698412699 
 data_test logistic regression grid search accuracy:  0.8571428571428571 
 data_test decision tree accuracy:  0.8158730158730159
